# 换手率因子 (PMO) 完整教程：从异常换手率到多空组合检验

## 📚 教学目标
1. 理解**换手率因子**的定义：异常换手率 = 近 1 月均值 / 近 1 年均值，低换手率 − 高换手率 = PMO
2. 掌握 **A 股换手率因子**的构建方法（Liu et al. 2019）
3. 在**微型数据集**（10 只股票 × 1 月）上手算分组检验过程
4. 扩展到 **300 只股票 × 60 月**，检验低换手率溢价
5. 理解 **A 股低换手率溢价**的行为金融学解释

**参考书**：《因子投资：方法与实践》（石川）第 3.8 节
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到真实规模

---

## 1. 什么是换手率因子？为什么低换手率股票收益更高？

### 🎯 一个直觉问题

假设你面前有两只股票：A 股票每天换手率 5%（很活跃），B 股票每天换手率 0.5%（很冷清）。你更愿意买哪个？

研究发现，低换手率（冷清）的股票长期收益更高。**换手率因子 (PMO, Pessimistic-Minus-Optimistic)** 就是检验这个现象：做多低换手率股票、做空高换手率股票，能否获得显著的超额收益？

### 📖 书中的定义

> 换手率和预期收益之间呈负相关关系。

> Liu et al.（2019）指出 A 股市场换手率对股票未来收益有显著的负面影响并据此构建了换手率因子（Pessimistic-Minus-Optimistic，PMO）。

### 📐 换手率因子的理论基础

| 理论来源 | 核心逻辑 |
|---------|----------|
| **投资者情绪** | 换手率反映过度自信和盲目乐观 → 情绪指标 |
| **投资者注意力** | 高换手率 = 高关注度 → 估值偏高 |
| **意见分歧** | 高换手率 = 投资者意见分歧大 → 不确定性高 |
| **崩盘风险** | 高换手率 + 高 past return → 崩盘风险高 |
| **套利限制** | 换手率异象与行为偏差和套利限制更相关 |

### 📐 异常换手率的定义

$$\text{Abnormal Turnover}_{i,t} = \frac{\bar{\text{TO}}_{i,t}^{21}}{\bar{\text{TO}}_{i,t}^{252}}$$

其中：
- $\bar{\text{TO}}_{i,t}^{21}$ = 过去 21 个交易日的平均换手率
- $\bar{\text{TO}}_{i,t}^{252}$ = 过去 252 个交易日的平均换手率
- 异常换手率 > 1 表示近期换手率高于历史均值（乐观）
- 异常换手率 < 1 表示近期换手率低于历史均值（悲观）

### 📐 PMO 因子的定义

$$\text{PMO} = R_{\text{Low TO}} - R_{\text{High TO}}$$

其中：
- $R_{\text{Low TO}}$ = 低异常换手率组（悲观）的组合收益率
- $R_{\text{High TO}}$ = 高异常换手率组（乐观）的组合收益率
- PMO > 0 表示低换手率股票收益高于高换手率股票

In [ ]:
import sys, os
print(f"Python: {sys.executable}")
print(f"Version: {sys.version}")
try:
    import matplotlib
    print(f"matplotlib: {matplotlib.__version__}")
except ImportError:
    print("❌ matplotlib 未安装! 请运行: !pip install matplotlib seaborn statsmodels scipy")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 微型数据集：10 只股票的手算演示

### 🎯 场景

假设市场上只有 **10 只股票**，我们用异常换手率作为排序变量，检验低换手率股票是否获得更高收益。

### 📐 数据生成逻辑

$$r_i = \bar{r} + \gamma \cdot (1 - \text{AbnormalTO}_i) + \varepsilon_i$$

其中：
- $\gamma$ = 换手率效应系数（> 0 表示低换手率带来高收益）
- $\varepsilon_i$ = 个股噪声

In [ ]:
# ========== 构造 10 只股票的微型数据 ==========
np.random.seed(42)

stocks = ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10']

# 异常换手率：近期/历史（从低到高）
abnormal_to = np.array([0.3, 0.5, 0.6, 0.7, 0.8, 1.0, 1.2, 1.5, 2.0, 3.0])

# 数据生成参数
base_return = 1.0    # 月基础收益率 1%
gamma = 2.0          # 换手率效应：低换手率 → 高收益
noise = np.random.normal(0, 2.0, 10)

# 收益率 = 基础 + 换手率效应 + 噪声（低换手率收益更高）
returns = base_return + gamma * (1 - abnormal_to) + noise

# 构建 DataFrame
df_micro = pd.DataFrame({
    'Stock': stocks,
    'Abnormal TO': abnormal_to,
    'Return (%)': np.round(returns, 2)
})

print("📋 10 只股票数据集：")
print(df_micro.to_string(index=False))
print(f"\n💡 换手率效应系数 γ = {gamma}")
print(f"   异常换手率 < 1 = 近期比历史冷清（悲观），> 1 = 近期比历史活跃（乐观）")

### 📐 步骤 1: 按异常换手率排序分组

将 10 只股票按异常换手率从低到高排序，分为 2 组：
- **Low TO 组（Q1）**：异常换手率最低的 5 只（悲观）→ 做多
- **High TO 组（Q2）**：异常换手率最高的 5 只（乐观）→ 做空

$$\text{PMO} = \bar{R}_{\text{Low TO}} - \bar{R}_{\text{High TO}}$$

In [ ]:
# ========== 步骤 1: 按异常换手率排序分组 ==========
print("📊 步骤 1: 按异常换手率排序分组")
print("─" * 55)

df_sorted = df_micro.sort_values('Abnormal TO').reset_index(drop=True)
df_sorted['Group'] = ['Low TO'] * 5 + ['High TO'] * 5

print("\n  Low TO 组（低换手率，做多）:")
for _, row in df_sorted[df_sorted['Group'] == 'Low TO'].iterrows():
    print(f"    {row['Stock']}: 异常TO = {row['Abnormal TO']:.1f},  收益 = {row['Return (%)']:>6.2f}%")

print("\n  High TO 组（高换手率，做空）:")
for _, row in df_sorted[df_sorted['Group'] == 'High TO'].iterrows():
    print(f"    {row['Stock']}: 异常TO = {row['Abnormal TO']:.1f},  收益 = {row['Return (%)']:>6.2f}%")

In [ ]:
# ========== 步骤 2: 计算各组平均收益率和 Spread ==========
print("📊 步骤 2: 计算各组平均收益率和 Spread")
print("─" * 55)

low_to_mean = df_sorted[df_sorted['Group'] == 'Low TO']['Return (%)'].mean()
high_to_mean = df_sorted[df_sorted['Group'] == 'High TO']['Return (%)'].mean()
spread = low_to_mean - high_to_mean

print(f"\n  Low TO 组平均收益率  = {low_to_mean:.2f}%")
print(f"  High TO 组平均收益率 = {high_to_mean:.2f}%")
print(f"\n  📐 PMO = Low TO - High TO = {low_to_mean:.2f}% - {high_to_mean:.2f}% = {spread:.2f}%")
print(f"\n  💡 解读：低换手率股票比高换手率股票{'多' if spread > 0 else '少'}赚 {abs(spread):.2f}%")

---

## 3. 扩展到完整模拟：300 只股票 × 60 月

### 📐 数据生成过程 (DGP)

每月生成 300 只股票的截面数据：

$$r_{i,t} = \bar{r}_t + \gamma \cdot (1 - \text{AbnormalTO}_{i,t}) + \varepsilon_{i,t}$$

其中：
- $\gamma$ = 换手率效应强度（A 股中较强）
- $\varepsilon_{i,t} \sim N(0, 3)$ = 个股噪声

In [ ]:
# ========== 完整模拟参数 ==========
N_STOCKS = 300    # 每月 300 只股票
N_MONTHS = 60     # 60 个月
N_GROUPS = 5      # 分 5 组

print(f"📊 模拟参数：")
print(f"  股票数量: {N_STOCKS} 只/月")
print(f"  时间跨度: {N_MONTHS} 个月")
print(f"  分组数量: {N_GROUPS} 组")

In [ ]:
# ========== 生成模拟数据 ==========
np.random.seed(42)

all_data = []
for t in range(N_MONTHS):
    # 异常换手率：对数正态分布
    abnormal_to = np.random.lognormal(mean=0.0, sigma=0.5, size=N_STOCKS)
    abnormal_to = np.clip(abnormal_to, 0.1, 5.0)
    
    # 基础收益率（每月不同）
    base_return = np.random.normal(1.0, 0.5)
    
    # 换手率效应：低换手率收益更高
    gamma = 1.5
    to_effect = gamma * (1 - abnormal_to)
    
    # 噪声
    noise = np.random.normal(0, 3.0, N_STOCKS)
    
    # 收益率
    returns = base_return + to_effect + noise
    
    month_df = pd.DataFrame({
        'Month': t + 1,
        'Stock': [f'S{i:03d}' for i in range(N_STOCKS)],
        'AbnormalTO': abnormal_to,
        'Return': returns
    })
    all_data.append(month_df)

df_all = pd.concat(all_data, ignore_index=True)

print(f"✅ 数据生成完成：{len(df_all)} 条记录")
print(f"   股票数: {df_all['Stock'].nunique()}")
print(f"   月份数: {df_all['Month'].nunique()}")
print(f"\n📊 异常换手率分布统计：")
print(df_all['AbnormalTO'].describe().round(4))

### 📐 步骤 3: 每月按异常换手率分组

每月末将股票按异常换手率从低到高分成 5 组：
- Q1 = 异常换手率最低（Low TO，悲观）
- Q5 = 异常换手率最高（High TO，乐观）

$$\text{PMO}_t = \bar{R}_{Q1,t} - \bar{R}_{Q5,t}$$

In [ ]:
# ========== 每月按异常换手率分组 ==========
def assign_to_group(month_df):
    """按异常换手率从低到高分 5 组"""
    month_df = month_df.copy()
    month_df['TOGroup'] = pd.qcut(month_df['AbnormalTO'], N_GROUPS, 
                                   labels=['Q1(Low)', 'Q2', 'Q3', 'Q4', 'Q5(High)'])
    return month_df

df_all = df_all.groupby('Month', group_keys=False).apply(assign_to_group)

# 计算每月各组平均收益率
monthly_group_returns = df_all.groupby(['Month', 'TOGroup'])['Return'].mean().unstack()

# 计算每月 PMO = Q1 - Q5
monthly_spreads = monthly_group_returns['Q1(Low)'] - monthly_group_returns['Q5(High)']

print("📊 每月各组平均收益率（前 5 个月）：")
print(monthly_group_returns.head().round(3))
print(f"\n📊 每月 PMO Spread（前 5 个月）：")
print(monthly_spreads.head().round(3))

### 📐 步骤 4: T 检验——PMO 是否显著大于零？

$$t = \frac{\overline{\text{PMO}}}{s_{\text{PMO}} / \sqrt{T}}$$

In [ ]:
# ========== T 检验 ==========
spreads = monthly_spreads.values
spread_mean = spreads.mean()
spread_std = spreads.std(ddof=1)
spread_se = spread_std / np.sqrt(N_MONTHS)
t_stat = spread_mean / spread_se
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=N_MONTHS - 1))

print("📊 步骤 4: T 检验结果")
print("─" * 55)
print(f"  PMO 均值       = {spread_mean:.4f}%")
print(f"  PMO 标准差     = {spread_std:.4f}%")
print(f"  标准误         = {spread_se:.4f}%")
print(f"  T 统计量       = {t_stat:.4f}")
print(f"  P 值 (双尾)    = {p_value:.6f}")
print(f"\n  💡 解读：")
if abs(t_stat) > 2.6:
    print(f"  ✓ |t| = {abs(t_stat):.2f} > 2.6 → 在 1% 水平下显著")
elif abs(t_stat) > 2.0:
    print(f"  ✓ |t| = {abs(t_stat):.2f} > 2.0 → 在 5% 水平下显著")
else:
    print(f"  ✗ |t| = {abs(t_stat):.2f} < 2.0 → 不显著")

In [ ]:
# ========== 用 scipy 验证 ==========
t_scipy, p_scipy = stats.ttest_1samp(spreads, 0)

print("🔬 scipy.stats.ttest_1samp 验证:")
print(f"  手算 T 统计量 = {t_stat:.6f}")
print(f"  scipy T 统计量 = {t_scipy:.6f}")
print(f"  手算 P 值     = {p_value:.6f}")
print(f"  scipy P 值    = {p_scipy:.6f}")
print(f"\n  ✅ 验证通过！")

### 📐 步骤 5: 经济意义——夏普比率

$$\text{SR}_{\text{monthly}} = \frac{\overline{\text{PMO}}}{s_{\text{PMO}}}$$

$$\text{SR}_{\text{annual}} = \text{SR}_{\text{monthly}} \times \sqrt{12}$$

In [ ]:
# ========== 夏普比率 ==========
sr_monthly = spread_mean / spread_std
sr_annual = sr_monthly * np.sqrt(12)

print("📊 步骤 5: 夏普比率")
print("─" * 55)
print(f"  月度夏普比率 = {sr_monthly:.4f}")
print(f"  年化夏普比率 = {sr_annual:.4f}")
print(f"\n  📐 验证: t = SR_monthly × √T = {sr_monthly:.4f} × √{N_MONTHS} = {sr_monthly * np.sqrt(N_MONTHS):.4f}")
print(f"\n  💡 解读：书中实证显示换手率因子 t 值高达 5.45（等权），是 A 股最显著的因子之一")

---

## 4. 单调性检验：各组收益率是否单调递减？

In [ ]:
# ========== 单调性检验 ==========
avg_group_returns = monthly_group_returns.mean()
group_ranks = np.arange(1, N_GROUPS + 1)
group_ret_values = avg_group_returns.values

sp_corr, sp_p = stats.spearmanr(group_ranks, group_ret_values)

print("📊 单调性检验结果")
print("─" * 55)
print(f"  各组平均收益率：")
for i, (group, ret) in enumerate(avg_group_returns.items()):
    print(f"    {group}: {ret:.4f}%")
print(f"\n  Spearman 秩相关系数 = {sp_corr:.4f}")
print(f"  P 值 = {sp_p:.6f}")
print(f"\n  💡 解读：")
if sp_corr < -0.8:
    print(f"  ✓ ρ = {sp_corr:.2f} < -0.8 → 强负单调性（收益率随换手率上升而下降）")
elif sp_corr < -0.5:
    print(f"  ✓ ρ = {sp_corr:.2f} < -0.5 → 中等负单调性")
else:
    print(f"  ✗ ρ = {sp_corr:.2f} → 单调性不佳")

---

## 5. 可视化：换手率效应的全景图

In [ ]:
# ========== 可视化 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 图1: 各组收益柱状图 ---
ax1 = axes[0]
group_labels = avg_group_returns.index.tolist()
group_vals = avg_group_returns.values
colors = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, N_GROUPS))  # 反转颜色
bars = ax1.bar(group_labels, group_vals, color=colors, edgecolor='black', alpha=0.85)
for bar, v in zip(bars, group_vals):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{v:.3f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)
ax1.plot(group_labels, group_vals, 'ko--', linewidth=2, markersize=8, zorder=5)
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Abnormal TO Group (Q1=Low → Q5=High)', fontsize=11)
ax1.set_ylabel('Average Monthly Return (%)', fontsize=11)
ax1.set_title(f'Turnover Monotonicity (ρ = {sp_corr:.3f})', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# --- 图2: Spread 时间序列 ---
ax2 = axes[1]
months = range(1, N_MONTHS + 1)
ax2.bar(months, spreads, color=['#2ecc71' if s > 0 else '#e74c3c' for s in spreads],
        alpha=0.7, edgecolor='none')
ax2.axhline(y=spread_mean, color='blue', linestyle='--', linewidth=2,
            label=f'Mean = {spread_mean:.2f}%')
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_xlabel('Month', fontsize=11)
ax2.set_ylabel('PMO Spread (%)', fontsize=11)
ax2.set_title('Monthly PMO Spread (Low TO - High TO)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

# --- 图3: 累计收益率 ---
ax3 = axes[2]
cum_pmo = np.cumsum(spreads)
ax3.plot(range(1, N_MONTHS + 1), cum_pmo, 'b-', linewidth=2)
ax3.fill_between(range(1, N_MONTHS + 1), cum_pmo, alpha=0.3, color='steelblue')
ax3.axhline(y=0, color='black', linewidth=0.8)
ax3.set_xlabel('Month', fontsize=11)
ax3.set_ylabel('Cumulative PMO Return (%)', fontsize=11)
ax3.set_title('Cumulative PMO Factor Return', fontsize=13, fontweight='bold')
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  图1：各换手率组的平均月收益率，理想情况下应单调递减（低换手率收益最高）")
print(f"  图2：每月 PMO Spread，绿色为正（低换手率跑赢），红色为负")
print(f"  图3：PMO 因子的累计收益率曲线，书中实证显示该曲线非常平顺")

---

## 6. 结果汇总报告

In [ ]:
# ========== 汇总报告 ==========
print("=" * 60)
print("📋 换手率因子 (PMO) 实证分析报告")
print("=" * 60)

print(f"\n🎯 研究问题:")
print(f"   低换手率股票是否比高换手率股票获得更高收益？")

print(f"\n📊 数据概况:")
print(f"   股票数量: {N_STOCKS} 只/月")
print(f"   时间跨度: {N_MONTHS} 个月")
print(f"   排序变量: 异常换手率 = 近 21 日均值 / 近 252 日均值")

print(f"\n🧮 统计检验:")
print(f"   PMO 月均收益率 = {spread_mean:.4f}%")
print(f"   T 统计量      = {t_stat:.4f}")
print(f"   P 值           = {p_value:.6f}")
print(f"   月度夏普比率   = {sr_monthly:.4f}")
print(f"   年化夏普比率   = {sr_annual:.4f}")
print(f"   Spearman ρ    = {sp_corr:.4f}")

print(f"\n🎯 结论:")
if t_stat > 2.0:
    print(f"  ✓ 换手率效应显著：低换手率股票收益显著高于高换手率股票")
    print(f"  💡 书中实证：等权下 PMO 月均收益率 1.50%（t=5.45），是 A 股最显著的因子")
else:
    print(f"  ✗ 换手率效应不显著")

print(f"\n💡 实际应用注意：")
print(f"   - 空头端（高换手率组）对因子收益贡献大，但做空困难")
print(f"   - 多头端（低换手率组合）年化夏普比率 0.65，显著高于市场组合 0.40")
print(f"   - 可利用多头端跑赢市场的特点进行实际投资")

print("\n" + "=" * 60)

---

## 📌 核心概念回顾

### 📌 换手率因子 (PMO)
- **定义**: Pessimistic-Minus-Optimistic，做多低换手率、做空高换手率的多空组合
- **公式**: $\text{PMO} = R_{\text{Low TO}} - R_{\text{High TO}}$
- **构建**: 异常换手率 = 近 21 日均值 / 近 252 日均值
- **判断标准**: t > 2.0 表示显著，书中 t 值高达 5.45

### 📌 异常换手率的含义
- **< 1**: 近期比历史冷清（悲观，Pessimistic）
- **> 1**: 近期比历史活跃（乐观，Optimistic）
- **优点**: 与市值等变量相关性低，是"优秀"的因子

### 📌 A 股换手率效应的特殊性
- **最显著因子**: 书中 t 值 5.45（等权），远超规模和价值因子
- **持续性强**: 组合形成后 5 年内都可能获得显著超额收益
- **空头贡献大**: 高换手率组收益率明显更低

### 🔗 完整流程
```
换手率数据 → 计算异常换手率 → 每月排序分组 → PMO = Low - High
    ↓            ↓              ↓              ↓
 21日/252日     比值           5 组          T 检验 + 单调性
```

---

## ❌ 常见误区

### ❌ 误区 1: 换手率就是流动性
**✓ 正确理解**: Amihud (2002) 定义的非流动性才是流动性指标。换手率更多反映投资者情绪和意见分歧。

### ❌ 误区 2: 高换手率一定好
**✓ 正确理解**: A 股中低换手率股票收益更高（流动性溢价的反面）。高换手率反映过度乐观和投机。

### ❌ 误区 3: 用原始换手率就够了
**✓ 正确理解**: 应使用异常换手率（近期/历史），消除个股固有的换手率差异。

### ❌ 误区 4: 换手率因子的纸面收益就是实际收益
**✓ 正确理解**: 空头端（高换手率组）对收益贡献大，但做空困难，实际收益会打折。

### ❌ 误区 5: 换手率异象会被已知因子解释
**✓ 正确理解**: 书中指出换手率异象不能被 Fama-French 三因子模型等经典模型完全解释。